# Data Processing
Loading and cleaning 500,000 rows from US Accidents dataset using Pandas.

In [1]:
import pandas as pd
import numpy as np
import os
import time
print("Libraries loaded")

Libraries loaded


In [2]:
# Sample and save small dataset
# Purpose: Read full CSV but keep only 500,000 rows. This cell runs ONCE and saves accidents_sample.csv.
# After that, always load from accidents_sample.csv.

sample_path = "../data/accidents_sample.csv"

if os.path.exists(sample_path):
    print("Sample already exists, skipping")
else:
    start_time = time.time()
    # Read full CSV with slightly more rows
    df_full = pd.read_csv("../data/us_accidents.csv", nrows=600000)
    # Take random sample of 500,000 rows
    df_sample = df_full.sample(n=500000, random_state=42)
    # Save to CSV
    df_sample.to_csv(sample_path, index=False)
    elapsed = time.time() - start_time
    print(f"Total rows saved: {len(df_sample)}")
    print(f"Time taken: {elapsed:.2f} seconds")
    print("Sampling done. Future cells load from accidents_sample.csv")

Total rows saved: 500000
Time taken: 86.86 seconds
Sampling done. Future cells load from accidents_sample.csv


In [3]:
# Load sampled data
start_time = time.time()
df = pd.read_csv("../data/accidents_sample.csv")
load_time = time.time() - start_time
print(f"Shape: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(df.head(3))
print(f"Time taken to load: {load_time:.2f} seconds")

Shape: (500000, 46)
Columns: ['ID', 'Source', 'Severity', 'Start_Time', 'End_Time', 'Start_Lat', 'Start_Lng', 'End_Lat', 'End_Lng', 'Distance(mi)', 'Description', 'Street', 'City', 'County', 'State', 'Zipcode', 'Country', 'Timezone', 'Airport_Code', 'Weather_Timestamp', 'Temperature(F)', 'Wind_Chill(F)', 'Humidity(%)', 'Pressure(in)', 'Visibility(mi)', 'Wind_Direction', 'Wind_Speed(mph)', 'Precipitation(in)', 'Weather_Condition', 'Amenity', 'Bump', 'Crossing', 'Give_Way', 'Junction', 'No_Exit', 'Railway', 'Roundabout', 'Station', 'Stop', 'Traffic_Calming', 'Traffic_Signal', 'Turning_Loop', 'Sunrise_Sunset', 'Civil_Twilight', 'Nautical_Twilight', 'Astronomical_Twilight']
         ID   Source  Severity           Start_Time             End_Time  \
0    A-4243  Source2         3  2016-07-26 09:41:17  2016-07-26 10:11:17   
1   A-60609  Source2         3  2017-01-05 07:10:49  2017-01-05 07:44:00   
2  A-392843  Source2         2  2017-03-24 11:46:05  2017-03-24 12:30:30   

   Start_Lat   S

In [4]:
# Null analysis
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df)) * 100

print("=== Null counts per column ===")
print(null_counts[null_counts > 0].sort_values(ascending=False))

print("\n=== Columns with more than 20% nulls ===")
high_null_cols = null_pct[null_pct > 20]
print(high_null_cols.sort_values(ascending=False))

print("\n=== Null percentage for key columns ===")
key_columns = ['Severity', 'Start_Lat', 'Start_Lng', 'Start_Time', 'Weather_Condition', 'Temperature(F)']
for col in key_columns:
    if col in df.columns:
        pct = null_pct.get(col, 0)
        print(f"{col}: {pct:.2f}%")

=== Null counts per column ===
End_Lat                  500000
End_Lng                  500000
Precipitation(in)        385136
Wind_Chill(F)            377094
Wind_Speed(mph)           77548
Visibility(mi)            10437
Weather_Condition         10056
Humidity(%)                8567
Temperature(F)             7839
Pressure(in)               6038
Wind_Direction             5153
Weather_Timestamp          4455
Street                      729
Nautical_Twilight           594
Astronomical_Twilight       594
Sunrise_Sunset              594
Civil_Twilight              594
Airport_Code                257
Timezone                    110
Zipcode                      49
City                         20
dtype: int64

=== Columns with more than 20% nulls ===
End_Lat              100.0000
End_Lng              100.0000
Precipitation(in)     77.0272
Wind_Chill(F)         75.4188
dtype: float64

=== Null percentage for key columns ===
Severity: 0.00%
Start_Lat: 0.00%
Start_Lng: 0.00%
Start_Time: 0.00

In [5]:
# Data cleaning — do in this exact order
print(f"Initial row count: {len(df)}")

# a) Drop rows where Severity is null
df = df.dropna(subset=['Severity'])
print(f"✅ a) Dropped rows with null Severity: {len(df)}")

# b) Drop rows where Start_Lat or Start_Lng is null
df = df.dropna(subset=['Start_Lat', 'Start_Lng'])
print(f"✅ b) Dropped rows with null Start_Lat/Start_Lng: {len(df)}")

# c) Drop rows where Start_Time is null
df = df.dropna(subset=['Start_Time'])
print(f"✅ c) Dropped rows with null Start_Time: {len(df)}")

# d) Drop columns where null % > 40%
null_pct = (df.isnull().sum() / len(df)) * 100
cols_to_drop = null_pct[null_pct > 40].index.tolist()
df = df.drop(columns=cols_to_drop)
print(f"✅ d) Dropped columns with >40% nulls: {cols_to_drop}, remaining columns: {len(df.columns)}")

# e) Fill numeric nulls with median
numeric_cols = ['Temperature(F)', 'Humidity(%)', 'Visibility(mi)', 'Wind_Speed(mph)', 'Precipitation(in)']
for col in numeric_cols:
    if col in df.columns:
        median_val = df[col].median()
        df[col] = df[col].fillna(median_val)
        print(f"   Filled {col} nulls with median: {median_val}")
print(f"✅ e) Filled numeric nulls with median")

# f) Fill Weather_Condition nulls with "Unknown"
if 'Weather_Condition' in df.columns:
    df['Weather_Condition'] = df['Weather_Condition'].fillna("Unknown")
    print(f"✅ f) Filled Weather_Condition nulls with 'Unknown'")

# g) Drop duplicates on ID column
if 'ID' in df.columns:
    df = df.drop_duplicates(subset=['ID'])
    print(f"✅ g) Dropped duplicates on ID: {len(df)}")

# h) Convert Start_Time and End_Time to datetime
df['Start_Time'] = pd.to_datetime(df['Start_Time'], errors='coerce')
if 'End_Time' in df.columns:
    df['End_Time'] = pd.to_datetime(df['End_Time'], errors='coerce')
print(f"✅ h) Converted Start_Time and End_Time to datetime")

# i) Filter: Start_Lat between 24-50, Start_Lng between -125 and -66
df = df[(df['Start_Lat'] >= 24) & (df['Start_Lat'] <= 50)]
df = df[(df['Start_Lng'] >= -125) & (df['Start_Lng'] <= -66)]
print(f"✅ i) Filtered to US bounds (lat 24-50, lng -125 to -66): {len(df)}")

print(f"\nFinal row count: {len(df)}")

Initial row count: 500000
✅ a) Dropped rows with null Severity: 500000
✅ b) Dropped rows with null Start_Lat/Start_Lng: 500000
✅ c) Dropped rows with null Start_Time: 500000
✅ d) Dropped columns with >40% nulls: ['End_Lat', 'End_Lng', 'Wind_Chill(F)', 'Precipitation(in)'], remaining columns: 42
   Filled Temperature(F) nulls with median: 69.1
   Filled Humidity(%) nulls with median: 66.0
   Filled Visibility(mi) nulls with median: 10.0
   Filled Wind_Speed(mph) nulls with median: 8.0
✅ e) Filled numeric nulls with median
✅ f) Filled Weather_Condition nulls with 'Unknown'
✅ g) Dropped duplicates on ID: 500000
✅ h) Converted Start_Time and End_Time to datetime
✅ i) Filtered to US bounds (lat 24-50, lng -125 to -66): 500000

Final row count: 500000


In [6]:
# Basic aggregations — use pandas groupby

# a) Accidents per State — top 15, sorted descending
print("=== a) Accidents per State (Top 15) ===")
state_counts = df['State'].value_counts().head(15)
print(state_counts)

# b) Accidents per Severity with percentage
print("\n=== b) Accidents per Severity ===")
severity_counts = df['Severity'].value_counts().sort_index()
severity_pct = (severity_counts / len(df)) * 100
for sev, cnt in severity_counts.items():
    print(f"Severity {sev}: {cnt:,} ({severity_pct[sev]:.2f}%)")

# c) Average severity by Weather_Condition — top 15, filter conditions with count > 50
print("\n=== c) Average Severity by Weather Condition (Top 15, count > 50) ===")
weather_stats = df.groupby('Weather_Condition').agg(
    avg_severity=('Severity', 'mean'),
    count=('Severity', 'count')
).reset_index()
weather_stats = weather_stats[weather_stats['count'] > 50].sort_values('avg_severity', ascending=False).head(15)
print(weather_stats.to_string(index=False))

# d) Accidents by hour of day (extract from Start_Time)
print("\n=== d) Accidents by Hour of Day ===")
df['Hour'] = df['Start_Time'].dt.hour
hour_counts = df['Hour'].value_counts().sort_index()
print(hour_counts)

# e) Accidents by day of week
print("\n=== e) Accidents by Day of Week ===")
df['DayOfWeek'] = df['Start_Time'].dt.day_name()
dow_counts = df['DayOfWeek'].value_counts()
# Reorder by day of week
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
dow_counts = dow_counts.reindex(day_order)
print(dow_counts)

=== a) Accidents per State (Top 15) ===
State
CA    157760
TX     73888
FL     48940
PA     29291
NY     24005
MI     23425
IL     17907
GA     17900
OH     12854
WA     11239
MD      9691
VA      9535
SC      9378
NJ      8890
MA      8638
Name: count, dtype: int64

=== b) Accidents per Severity ===
Severity 1: 24,061 (4.81%)
Severity 2: 289,701 (57.94%)
Severity 3: 185,465 (37.09%)
Severity 4: 773 (0.15%)

=== c) Average Severity by Weather Condition (Top 15, count > 50) ===
           Weather_Condition  avg_severity  count
                  Heavy Snow      2.585034    147
         Light Freezing Rain      2.555556    180
                        Snow      2.482353    340
      Thunderstorms and Rain      2.457584    389
                     Drizzle      2.447368    190
                        Haze      2.431908   5309
                  Heavy Rain      2.431373   1377
Heavy Thunderstorms and Rain      2.430809    383
                        Rain      2.430728   4107
                  

In [7]:
# Save cleaned data
output_path = "../data/accidents_clean.parquet"
df.to_parquet(output_path, index=False)

import os
file_size_mb = os.path.getsize(output_path) / (1024 * 1024)
print(f"File size: {file_size_mb:.2f} MB")
print(f"Row count: {len(df)}")
print(f"Column count: {len(df.columns)}")
print("Saved successfully")

File size: 44.57 MB
Row count: 500000
Column count: 44
Saved successfully


In [8]:
# Verify cleaned data
df_verify = pd.read_parquet("../data/accidents_clean.parquet")
print(f"Verified row count: {len(df_verify)}")
print(f"Expected row count: {len(df)}")
print(f"Row count matches: {len(df_verify) == len(df)}")
print("\n=== Schema (dtypes) ===")
print(df_verify.dtypes)
print("\nVerification passed")

Verified row count: 500000
Expected row count: 500000
Row count matches: True

=== Schema (dtypes) ===
ID                                  str
Source                              str
Severity                          int64
Start_Time               datetime64[us]
End_Time                 datetime64[us]
Start_Lat                       float64
Start_Lng                       float64
Distance(mi)                    float64
Description                         str
Street                              str
City                                str
County                              str
State                               str
Zipcode                             str
Country                             str
Timezone                            str
Airport_Code                        str
Weather_Timestamp                   str
Temperature(F)                  float64
Humidity(%)                     float64
Pressure(in)                    float64
Visibility(mi)                  float64
Wind_Direction   

In [9]:
print(f"Total rows after cleaning: {len(df)}")
print(f"Columns kept: {len(df.columns)}")
print("File saved: accidents_clean.parquet")
print("Next: Feature Engineering notebook")

Total rows after cleaning: 500000
Columns kept: 44
File saved: accidents_clean.parquet
Next: Feature Engineering notebook
